# 🧪 Summarization Evaluation Pipeline: Submission vs. Ground Truth

This script aligns your model's generated summaries with the actual ground truth and calculates the required NLP metrics: ROUGE-L and BERTScore.

In [ ]:
import json
import codecs
from pathlib import Path
import numpy as np
import string
from datetime import datetime
from rouge_score import rouge_scorer
from bert_score import score as calc_bert_score

In [ ]:
BASE_DIR = Path.cwd().parent
EVAL_CATEGORY = "dev"
EVAL_DATA_DIR = (
    BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data" / EVAL_CATEGORY
)

SUBMISSION_FILE = BASE_DIR / "submission_finetuning_summary_dev.json"
RESULTS_FILE = BASE_DIR / "evaluation_summarization_results.json"

In [ ]:
def normalize_text(text):
    if not text:
        return ""

    # 1. Handle unicode-escaped sequences
    try:
        text = codecs.decode(text, "unicode_escape")
    except:
        pass

    # 2. Lowercase and strip
    text = text.strip().lower()

    # 3. Basic punctuation removal (as you already have)
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Note: For "unit conversion," you might need specific regex if
    # the task expects "100C" to match "100 degrees celsius"
    return text

Load Predictions into a searchable dictionary: `preds_dict[sample_id][subfigure] = summary`

In [ ]:
preds_dict = {}

with open(SUBMISSION_FILE, "r", encoding="utf-8") as f:
    submission_data = json.load(f)
    for item in submission_data:
        s_id = item.get("sample_id")
        preds_dict[s_id] = {}

        # Extract the summarization dictionary
        summarizations = item.get("summarization", {})
        for sub_key, summary_text in summarizations.items():
            if summary_text:  # Only store non-empty predictions
                preds_dict[s_id][sub_key] = normalize_text(summary_text)

Load Ground Truth and match with Predictions

In [ ]:
preds_list = []
gts_list = []

json_files = list(EVAL_DATA_DIR.rglob("*.json"))
for json_file in json_files:
    fullpath = str(json_file)
    if ".vscode" in fullpath or "content" in fullpath:
        continue

    with open(json_file, "r", encoding="utf-8") as f:
        gt_data = json.load(f)

    sample_id = gt_data.get("sample_id")
    if not sample_id or sample_id not in preds_dict:
        continue

    # Extract ground truth summarizations
    gt_summarizations = gt_data.get("summarization", {})

    for sub_key, gt_summary in gt_summarizations.items():
        gt_summary = normalize_text(gt_summary)

        # Skip if ground truth is empty or missing
        if not gt_summary:
            continue

        # Match with prediction if available
        pred_summary = preds_dict[sample_id].get(sub_key, "")

        preds_list.append(pred_summary)
        gts_list.append(gt_summary)

Calculate Evaluation Metrics (ROUGE-L and BERTScore)

In [ ]:
results = {}

if preds_list and gts_list:
    print("\n🧮 Calculating metrics...")

    # 1. Calculate ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [
        scorer.score(gt, pred)["rougeL"].fmeasure
        for pred, gt in zip(preds_list, gts_list)
    ]
    results["ROUGE-L F1"] = np.mean(rouge_scores)
    print("✅ Computed ROUGE-L metrics.")

    # 2. Calculate BERTScore
    # We use distilbert-base-uncased as a lightweight default, but you can
    # upgrade to roberta-large for more robust semantic similarity if needed.
    P, R, F1 = calc_bert_score(
        preds_list,
        gts_list,
        lang="en",
        verbose=False,
        model_type="distilbert-base-uncased",
    )
    results["BERTScore F1"] = F1.mean().item()
    results["BERTScore Precision"] = P.mean().item()
    results["BERTScore Recall"] = R.mean().item()
    print("📋 Computed BERTScore metrics.")
else:
    print("⚠️ No matching pairs found to evaluate.")

Append the evaluation metrics to the JSON file to keep a running history

In [ ]:
if RESULTS_FILE.exists():
    with open(RESULTS_FILE, "r") as f:
        try:
            history = json.load(f)
        except json.JSONDecodeError:
            history = []
else:
    history = []

history.append(
    {
        "run_type": "submission_evaluation",
        "task": "Summarization",
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "submission_file": SUBMISSION_FILE.name,
        "sample_size": len(preds_list),
        "metrics": results,
    }
)

with open(RESULTS_FILE, "w") as f:
    json.dump(history, f, indent=4)

print(f"\n✨ Evaluation complete! State updated in: {RESULTS_FILE}")
print(json.dumps(results, indent=2))